# K-Means Regional Clustering (Non-Spatial)

K-Means applied to the same two feature sets used in `skater_clustering.ipynb`, **without** any spatial constraint.  The comparison isolates the effect of geographic contiguity: SKATER pays a feature-space cost to keep regions spatially connected; K-Means shows the floor — the best within-cluster homogeneity achievable with no spatial budget.

| # | Features | Interpretation |
|---|----------|----------------|
| 1 | `lp3_scale`, `lp3_skew` | Shape of the flood frequency curve |
| 2 | `log10(1/aep)` for action / flood / moderate / major | How rare local flood impacts are |

The final section loads saved SKATER labels and computes ARI, NMI, and a spatial disagreement map for each analysis.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import rplot

plt.style.use('ryan')

from pathlib import Path
from shapely.geometry import Point
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from scipy.optimize import linear_sum_assignment

## Configuration

In [ ]:
DATA_DIR   = Path("/home/ryan/data/flood_hazard")
GAGES2_DIR = Path("/home/ryan/data/usgs/GAGES_2/basinchar_and_report_sept_2011/spreadsheets-in-csv-format")
SKATER_DIR = Path(".")

N_CLUSTERS_LP3    = 8
N_CLUSTERS_AEP    = 8
MAX_DISTURB_INDEX = 15
WINSOR            = 0.02
RANDOM_STATE      = 42
N_INIT            = 20   # K-Means restarts
SAVEFIG           = False

CONUS_EXTENT = [-125, -66, 24, 50]

def make_conus_ax(fig, pos=111, title=''):
    subplot_args = pos if isinstance(pos, tuple) else (pos,)
    ax = fig.add_subplot(*subplot_args, projection=ccrs.AlbersEqualArea(
        central_longitude=-96, central_latitude=37.5))
    ax.set_extent(CONUS_EXTENT, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND,      facecolor='#f5f5f0', zorder=0)
    ax.add_feature(cfeature.OCEAN,     facecolor='#c8e0f0', zorder=0)
    ax.add_feature(cfeature.LAKES,     facecolor='#c8e0f0', zorder=1, alpha=0.6)
    ax.add_feature(cfeature.STATES,    linewidth=0.4, zorder=2, edgecolor='gray')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6, zorder=3)
    ax.add_feature(cfeature.BORDERS,   linewidth=0.6, zorder=3)
    if title:
        ax.set_title(title)
    return ax

## Load shared data

In [ ]:
ffa  = pd.read_parquet(DATA_DIR / "ffa" / "flood_frequency.parquet")
meta = pd.read_parquet(DATA_DIR / "metadata" / "site_info.parquet")[["site_no", "latitude", "longitude"]]

gages2 = pd.read_csv(GAGES2_DIR / "conterm_bas_classif.txt", encoding="latin1")
gages2["site_no"] = gages2["STAID"].astype(str).str.zfill(8)

print(f"FFA records : {len(ffa):,}")
print(f"Site meta   : {len(meta):,}")
print(f"GAGES-2     : {len(gages2):,}")

---
# Analysis 1: LP3 K-Means

Features: **`lp3_scale`** and **`lp3_skew`** — same filtering, winsorization, and RobustScaler as the SKATER notebook.

In [ ]:
df1 = (
    ffa[ffa.record_ok & ffa.lp3_scale.notna() & ffa.lp3_skew.notna()]
    [["site_no", "lp3_scale", "lp3_skew"]]
    .merge(meta, on="site_no")
    .merge(gages2[["site_no", "HYDRO_DISTURB_INDX"]], on="site_no", how="left")
)
df1 = df1[
    df1.HYDRO_DISTURB_INDX.notna() & (df1.HYDRO_DISTURB_INDX <= MAX_DISTURB_INDEX)
].reset_index(drop=True)

print(f"LP3 analysis: {len(df1):,} sites")

## Winsorize and scale features

In [ ]:
LP3_FEATS  = ["lp3_scale", "lp3_skew"]
LP3_SCALED = [f + "_s" for f in LP3_FEATS]

df1_clip = df1.copy()
for col in LP3_FEATS:
    lo, hi = df1[col].quantile([WINSOR, 1 - WINSOR])
    df1_clip[col] = df1[col].clip(lo, hi)
    print(f"{col}: clipped [{lo:.4f}, {hi:.4f}]")

scaler1 = RobustScaler()
X1 = scaler1.fit_transform(df1_clip[LP3_FEATS])
for i, col in enumerate(LP3_SCALED):
    df1[col] = X1[:, i]

## Silhouette sweep — LP3

Sweep k=2–14 and record mean silhouette score.

In [ ]:
sil_k_range  = range(2, 15)
sil_scores1  = []
inertias1    = []

for k in sil_k_range:
    km = KMeans(n_clusters=k, n_init=N_INIT, random_state=RANDOM_STATE)
    labels = km.fit_predict(X1)
    sil_scores1.append(silhouette_score(X1, labels))
    inertias1.append(km.inertia_)
    print(f"k={k:2d}  sil={sil_scores1[-1]:.4f}  inertia={inertias1[-1]:.1f}")

best_k1 = list(sil_k_range)[np.argmax(sil_scores1)]
print(f"\nBest k by silhouette: {best_k1}  (score={max(sil_scores1):.4f})")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(list(sil_k_range), sil_scores1, marker='o', color=rplot.OKABE_ITO[0])
axes[0].axvline(best_k1, color='red', linestyle='--', label=f'best k={best_k1}')
axes[0].axvline(N_CLUSTERS_LP3, color='gray', linestyle=':', label=f'SKATER k={N_CLUSTERS_LP3}')
axes[0].set_xlabel('Number of clusters')
axes[0].set_ylabel('Mean silhouette score')
axes[0].set_title('Silhouette — LP3 K-Means')
axes[0].legend()

axes[1].plot(list(sil_k_range), inertias1, marker='o', color=rplot.OKABE_ITO[1])
axes[1].axvline(best_k1, color='red', linestyle='--', label=f'best k={best_k1}')
axes[1].axvline(N_CLUSTERS_LP3, color='gray', linestyle=':', label=f'SKATER k={N_CLUSTERS_LP3}')
axes[1].set_xlabel('Number of clusters')
axes[1].set_ylabel('Inertia (within-cluster SS)')
axes[1].set_title('Elbow — LP3 K-Means')
axes[1].legend()

rplot.panel_labels(list(axes))
plt.tight_layout()
plt.show()

## K-Means clustering — LP3

Using `N_CLUSTERS_LP3` to match SKATER for a direct comparison.

In [ ]:
km1 = KMeans(n_clusters=N_CLUSTERS_LP3, n_init=N_INIT, random_state=RANDOM_STATE)
df1["cluster"] = km1.fit_predict(X1) + 1

sizes1 = df1["cluster"].value_counts().sort_index()
print(f"N_CLUSTERS={N_CLUSTERS_LP3}  |  silhouette={silhouette_score(X1, df1['cluster']):+.4f}")
print(sizes1.to_string())
print(f"\nMin: {sizes1.min()}  |  Max: {sizes1.max()}  |  Ratio: {sizes1.max()/sizes1.min():.1f}x")

## Cluster map — LP3

In [ ]:
cluster_colors1 = rplot.cluster_cmap(N_CLUSTERS_LP3).colors

fig = plt.figure(figsize=(16, 9))
ax  = make_conus_ax(fig, title=(
    f'LP3 K-Means clusters  (N={N_CLUSTERS_LP3})\n'
    f'Features: lp3_scale, lp3_skew'
))

for cl in sorted(df1["cluster"].unique()):
    idx = df1["cluster"] == cl
    ax.scatter(
        df1.loc[idx, "longitude"], df1.loc[idx, "latitude"],
        s=20, color=cluster_colors1[cl - 1], zorder=4,
        transform=ccrs.PlateCarree(),
        label=f'C{cl} (n={idx.sum():,})'
    )

ax.legend(markerscale=2, fontsize=7, ncol=2, loc='lower left', framealpha=0.8)
plt.tight_layout()
plt.show()

## Cluster profiles — LP3

In [ ]:
profile1 = (
    df1.groupby("cluster")[["lp3_scale", "lp3_skew"]]
    .agg(["mean", "std", "count"])
)
profile1.columns = ["_".join(c) for c in profile1.columns]
print(profile1.to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for i, feat in enumerate(LP3_FEATS):
    means = profile1[f"{feat}_mean"]
    stds  = profile1[f"{feat}_std"]
    axes[i].bar(
        means.index, means.values,
        yerr=stds.values, capsize=4,
        color=[cluster_colors1[c - 1] for c in means.index],
        edgecolor='none'
    )
    axes[i].set_xlabel('Cluster')
    axes[i].set_title(feat)
rplot.panel_labels(list(axes))
plt.suptitle('LP3 K-Means: cluster mean ± SD', y=1.01)
plt.tight_layout()
plt.show()

---
# Analysis 2: AEP Threshold K-Means

Features: **log₁₀ return period** for each of the four NWS flood threshold levels — action, flood, moderate, major.

In [ ]:
AEP_COLS = ["action_aep", "flood_aep", "moderate_aep", "major_aep"]

df2 = (
    ffa[ffa.record_ok & ~ffa.degenerate_fit]
    [["site_no"] + AEP_COLS]
    .dropna(subset=AEP_COLS)
    .merge(meta, on="site_no")
    .merge(gages2[["site_no", "HYDRO_DISTURB_INDX"]], on="site_no", how="left")
)
df2 = df2[
    df2.HYDRO_DISTURB_INDX.notna() & (df2.HYDRO_DISTURB_INDX <= MAX_DISTURB_INDEX)
].reset_index(drop=True)

print(f"AEP analysis: {len(df2):,} sites")

## Transform and scale features

In [ ]:
RP_FEATS  = ["action_rp", "flood_rp", "moderate_rp", "major_rp"]
RP_SCALED = [f + "_s" for f in RP_FEATS]

for aep_col, rp_col in zip(AEP_COLS, RP_FEATS):
    df2[rp_col] = np.log10(1.0 / df2[aep_col].clip(lower=1e-6))

df2_clip = df2.copy()
for col in RP_FEATS:
    lo, hi = df2[col].quantile([WINSOR, 1 - WINSOR])
    n_clipped = ((df2[col] < lo) | (df2[col] > hi)).sum()
    df2_clip[col] = df2[col].clip(lo, hi)
    print(f"{col}: clipped [{lo:.2f}, {hi:.2f}]  ({n_clipped} sites clipped)")

scaler2 = RobustScaler()
X2 = scaler2.fit_transform(df2_clip[RP_FEATS])
for i, col in enumerate(RP_SCALED):
    df2[col] = X2[:, i]

## Silhouette sweep — AEP thresholds

In [ ]:
sil_k_range  = range(2, 15)
sil_scores2  = []
inertias2    = []

for k in sil_k_range:
    km = KMeans(n_clusters=k, n_init=N_INIT, random_state=RANDOM_STATE)
    labels = km.fit_predict(X2)
    sil_scores2.append(silhouette_score(X2, labels))
    inertias2.append(km.inertia_)
    print(f"k={k:2d}  sil={sil_scores2[-1]:.4f}  inertia={inertias2[-1]:.1f}")

best_k2 = list(sil_k_range)[np.argmax(sil_scores2)]
print(f"\nBest k by silhouette: {best_k2}  (score={max(sil_scores2):.4f})")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(list(sil_k_range), sil_scores2, marker='o', color=rplot.OKABE_ITO[0])
axes[0].axvline(best_k2, color='red', linestyle='--', label=f'best k={best_k2}')
axes[0].axvline(N_CLUSTERS_AEP, color='gray', linestyle=':', label=f'SKATER k={N_CLUSTERS_AEP}')
axes[0].set_xlabel('Number of clusters')
axes[0].set_ylabel('Mean silhouette score')
axes[0].set_title('Silhouette — AEP K-Means')
axes[0].legend()

axes[1].plot(list(sil_k_range), inertias2, marker='o', color=rplot.OKABE_ITO[1])
axes[1].axvline(best_k2, color='red', linestyle='--', label=f'best k={best_k2}')
axes[1].axvline(N_CLUSTERS_AEP, color='gray', linestyle=':', label=f'SKATER k={N_CLUSTERS_AEP}')
axes[1].set_xlabel('Number of clusters')
axes[1].set_ylabel('Inertia')
axes[1].set_title('Elbow — AEP K-Means')
axes[1].legend()

rplot.panel_labels(list(axes))
plt.tight_layout()
plt.show()

## K-Means clustering — AEP thresholds

In [ ]:
km2 = KMeans(n_clusters=N_CLUSTERS_AEP, n_init=N_INIT, random_state=RANDOM_STATE)
df2["cluster"] = km2.fit_predict(X2) + 1

sizes2 = df2["cluster"].value_counts().sort_index()
print(f"N_CLUSTERS={N_CLUSTERS_AEP}  |  silhouette={silhouette_score(X2, df2['cluster']):+.4f}")
print(sizes2.to_string())
print(f"\nMin: {sizes2.min()}  |  Max: {sizes2.max()}  |  Ratio: {sizes2.max()/sizes2.min():.1f}x")

## Cluster map — AEP thresholds

In [ ]:
cluster_colors2 = rplot.cluster_cmap(N_CLUSTERS_AEP).colors

fig = plt.figure(figsize=(16, 9))
ax  = make_conus_ax(fig, title=(
    f'AEP threshold K-Means clusters  (N={N_CLUSTERS_AEP})\n'
    f'Features: log10 return period — action / flood / moderate / major'
))

for cl in sorted(df2["cluster"].unique()):
    idx = df2["cluster"] == cl
    ax.scatter(
        df2.loc[idx, "longitude"], df2.loc[idx, "latitude"],
        s=20, color=cluster_colors2[cl - 1], zorder=4,
        transform=ccrs.PlateCarree(),
        label=f'C{cl} (n={idx.sum():,})'
    )

ax.legend(markerscale=2, fontsize=7, ncol=2, loc='lower left', framealpha=0.8)
plt.tight_layout()
plt.show()

## Cluster profiles — AEP thresholds

In [ ]:
rp_medians2 = df2.groupby("cluster")[RP_FEATS].median()
rp_years2   = 10 ** rp_medians2
rp_years2.columns = [c.replace("_rp", "_yr") for c in rp_years2.columns]
print("Median return period (years) per cluster:")
print(rp_years2.round(1).to_string())

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
labels = ["Action", "Flood", "Moderate", "Major"]
for i, (rp, label) in enumerate(zip(RP_FEATS, labels)):
    for cl in sorted(df2["cluster"].unique()):
        vals = df2.loc[df2["cluster"] == cl, rp]
        axes[i].boxplot(
            vals, positions=[cl], widths=0.6,
            patch_artist=True,
            boxprops=dict(facecolor=cluster_colors2[cl - 1], alpha=0.8),
            medianprops=dict(color='black'),
            flierprops=dict(marker='.', markersize=2, alpha=0.3),
            whiskerprops=dict(linewidth=0.8),
            capprops=dict(linewidth=0.8)
        )
    axes[i].set_title(label)
    axes[i].set_xlabel('Cluster')
    axes[i].set_ylabel('log10(return period)')
rplot.panel_labels(list(axes))
plt.suptitle('AEP K-Means: log10 return period by cluster', y=1.01)
plt.tight_layout()
plt.show()

---
# Comparison 1: K-Means vs SKATER — LP3

Load saved SKATER LP3 labels and compare against the K-Means solution above.  The key question is:
- **Feature-space cost**: does SKATER's spatial constraint lower silhouette compared to K-Means at the same k?
- **Spatial pattern of disagreement**: do the sites that flip clusters under K-Means have a coherent geography (e.g., transition zones, isolated clusters)?

In [ ]:
def read_skater(path):
    df = pd.read_csv(path)[["site_no", "cluster"]]
    df["site_no"] = df["site_no"].astype(str).str.zfill(8)
    return df

skater_lp3 = read_skater(SKATER_DIR / f"site_regions_skater_lp3_{N_CLUSTERS_LP3}.csv")
skater_lp3 = skater_lp3.rename(columns={"cluster": "cl_skater"})

km_lp3 = df1[["site_no", "cluster"]].rename(columns={"cluster": "cl_km"})

cmp1 = km_lp3.merge(skater_lp3, on="site_no")
cmp1 = cmp1.merge(df1[["site_no", "latitude", "longitude"] + LP3_FEATS + LP3_SCALED], on="site_no")

print(f"Sites in K-Means  : {len(km_lp3):,}")
print(f"Sites in SKATER   : {len(skater_lp3):,}")
print(f"Sites in both     : {len(cmp1):,}")

In [ ]:
X1_cmp = cmp1[LP3_SCALED].values

sil_km     = silhouette_score(X1_cmp, cmp1["cl_km"])
sil_skater = silhouette_score(X1_cmp, cmp1["cl_skater"])
ari1       = adjusted_rand_score(cmp1["cl_skater"], cmp1["cl_km"])
nmi1       = normalized_mutual_info_score(cmp1["cl_skater"], cmp1["cl_km"])

print(f"Silhouette — K-Means : {sil_km:.4f}")
print(f"Silhouette — SKATER  : {sil_skater:.4f}  (spatial cost: {sil_km - sil_skater:+.4f})")
print(f"ARI                  : {ari1:.3f}   (0=random, 1=perfect)")
print(f"NMI                  : {nmi1:.3f}   (0=independent, 1=identical)")

## Side-by-side map — LP3

In [ ]:
n_cl_km1     = cmp1["cl_km"].nunique()
n_cl_sk1     = cmp1["cl_skater"].nunique()
cc_km1       = rplot.cluster_cmap(n_cl_km1).colors
cc_sk1       = rplot.cluster_cmap(n_cl_sk1).colors

fig = plt.figure(figsize=(13, 6))

ax1 = make_conus_ax(fig, pos=(1, 2, 1), title=f'LP3 K-Means  (N={N_CLUSTERS_LP3})')
for cl in sorted(cmp1["cl_km"].unique()):
    idx = cmp1["cl_km"] == cl
    ax1.scatter(
        cmp1.loc[idx, "longitude"], cmp1.loc[idx, "latitude"],
        s=6, color=cc_km1[cl - 1], zorder=4,
        transform=ccrs.PlateCarree(), label=f'C{cl}'
    )
ax1.legend(markerscale=2, fontsize=7, ncol=2, loc='lower left', framealpha=0.8)

ax2 = make_conus_ax(fig, pos=(1, 2, 2), title=f'LP3 SKATER  (N={N_CLUSTERS_LP3})')
for cl in sorted(cmp1["cl_skater"].unique()):
    idx = cmp1["cl_skater"] == cl
    ax2.scatter(
        cmp1.loc[idx, "longitude"], cmp1.loc[idx, "latitude"],
        s=6, color=cc_sk1[cl - 1], zorder=4,
        transform=ccrs.PlateCarree(), label=f'C{cl}'
    )
ax2.legend(markerscale=2, fontsize=7, ncol=2, loc='lower left', framealpha=0.8)

rplot.panel_labels([ax1, ax2])
plt.tight_layout()
plt.show()

## Spatial agreement map — LP3

Optimal label alignment via the Hungarian algorithm, then sites coloured by whether K-Means and SKATER agree.

In [ ]:
ct1 = pd.crosstab(cmp1["cl_skater"], cmp1["cl_km"],
                  rownames=["SKATER"], colnames=["K-Means"])
row_idx1, col_idx1 = linear_sum_assignment(-ct1.values)
sk_to_km1 = {ct1.index[r]: ct1.columns[c] for r, c in zip(row_idx1, col_idx1)}

cmp1["cl_skater_mapped"] = cmp1["cl_skater"].map(sk_to_km1)
cmp1["agree"] = cmp1["cl_skater_mapped"] == cmp1["cl_km"]

n_agree1 = cmp1["agree"].sum()
print(f"Overall agreement: {n_agree1:,}/{len(cmp1):,}  ({100*n_agree1/len(cmp1):.1f}%)")
print(f"\nOptimal SKATER → K-Means mapping:")
for sk, km in sorted(sk_to_km1.items()):
    n_ag = ct1.loc[sk, km] if km in ct1.columns else 0
    n_to = ct1.loc[sk].sum()
    print(f"  SKATER C{sk} → KM C{km}  ({n_ag:,}/{n_to:,} agree, {100*n_ag/n_to:.0f}%)")

agree1_df    = cmp1[cmp1["agree"]]
disagree1_df = cmp1[~cmp1["agree"]]

fig = plt.figure(figsize=(16, 9))
ax  = make_conus_ax(fig, title=(
    f"LP3: K-Means vs SKATER agreement (optimal label mapping)\n"
    f"ARI={ari1:.3f}  |  agree={n_agree1:,}/{len(cmp1):,} ({100*n_agree1/len(cmp1):.0f}%)  "
    f"|  sil cost={sil_km - sil_skater:+.4f}"
))
ax.scatter(
    agree1_df.longitude, agree1_df.latitude,
    s=15, color=rplot.OKABE_ITO[1], alpha=0.6,
    transform=ccrs.PlateCarree(), zorder=4,
    label=f"Agree ({len(agree1_df):,})"
)
ax.scatter(
    disagree1_df.longitude, disagree1_df.latitude,
    s=10, color=rplot.OKABE_ITO[5], alpha=0.9,
    transform=ccrs.PlateCarree(), zorder=5,
    label=f"Disagree ({len(disagree1_df):,})"
)
ax.legend(markerscale=2, fontsize=9, loc="lower left", framealpha=0.8)
plt.tight_layout()
plt.show()

---
# Comparison 2: K-Means vs SKATER — AEP thresholds

In [ ]:
skater_aep = read_skater(SKATER_DIR / f"site_regions_skater_aep_{N_CLUSTERS_AEP}.csv")
skater_aep = skater_aep.rename(columns={"cluster": "cl_skater"})

km_aep = df2[["site_no", "cluster"]].rename(columns={"cluster": "cl_km"})

cmp2 = km_aep.merge(skater_aep, on="site_no")
cmp2 = cmp2.merge(df2[["site_no", "latitude", "longitude"] + RP_FEATS + RP_SCALED], on="site_no")

print(f"Sites in K-Means  : {len(km_aep):,}")
print(f"Sites in SKATER   : {len(skater_aep):,}")
print(f"Sites in both     : {len(cmp2):,}")

In [ ]:
X2_cmp = cmp2[RP_SCALED].values

sil_km2     = silhouette_score(X2_cmp, cmp2["cl_km"])
sil_skater2 = silhouette_score(X2_cmp, cmp2["cl_skater"])
ari2        = adjusted_rand_score(cmp2["cl_skater"], cmp2["cl_km"])
nmi2        = normalized_mutual_info_score(cmp2["cl_skater"], cmp2["cl_km"])

print(f"Silhouette — K-Means : {sil_km2:.4f}")
print(f"Silhouette — SKATER  : {sil_skater2:.4f}  (spatial cost: {sil_km2 - sil_skater2:+.4f})")
print(f"ARI                  : {ari2:.3f}")
print(f"NMI                  : {nmi2:.3f}")

## Side-by-side map — AEP

In [ ]:
n_cl_km2 = cmp2["cl_km"].nunique()
n_cl_sk2 = cmp2["cl_skater"].nunique()
cc_km2   = rplot.cluster_cmap(n_cl_km2).colors
cc_sk2   = rplot.cluster_cmap(n_cl_sk2).colors

fig = plt.figure(figsize=(13, 6))

ax1 = make_conus_ax(fig, pos=(1, 2, 1), title=f'AEP K-Means  (N={N_CLUSTERS_AEP})')
for cl in sorted(cmp2["cl_km"].unique()):
    idx = cmp2["cl_km"] == cl
    ax1.scatter(
        cmp2.loc[idx, "longitude"], cmp2.loc[idx, "latitude"],
        s=6, color=cc_km2[cl - 1], zorder=4,
        transform=ccrs.PlateCarree(), label=f'C{cl}'
    )
ax1.legend(markerscale=2, fontsize=7, ncol=2, loc='lower left', framealpha=0.8)

ax2 = make_conus_ax(fig, pos=(1, 2, 2), title=f'AEP SKATER  (N={N_CLUSTERS_AEP})')
for cl in sorted(cmp2["cl_skater"].unique()):
    idx = cmp2["cl_skater"] == cl
    ax2.scatter(
        cmp2.loc[idx, "longitude"], cmp2.loc[idx, "latitude"],
        s=6, color=cc_sk2[cl - 1], zorder=4,
        transform=ccrs.PlateCarree(), label=f'C{cl}'
    )
ax2.legend(markerscale=2, fontsize=7, ncol=2, loc='lower left', framealpha=0.8)

rplot.panel_labels([ax1, ax2])
plt.tight_layout()
plt.show()

## Spatial agreement map — AEP

In [ ]:
ct2 = pd.crosstab(cmp2["cl_skater"], cmp2["cl_km"],
                  rownames=["SKATER"], colnames=["K-Means"])
row_idx2, col_idx2 = linear_sum_assignment(-ct2.values)
sk_to_km2 = {ct2.index[r]: ct2.columns[c] for r, c in zip(row_idx2, col_idx2)}

cmp2["cl_skater_mapped"] = cmp2["cl_skater"].map(sk_to_km2)
cmp2["agree"] = cmp2["cl_skater_mapped"] == cmp2["cl_km"]

n_agree2 = cmp2["agree"].sum()
print(f"Overall agreement: {n_agree2:,}/{len(cmp2):,}  ({100*n_agree2/len(cmp2):.1f}%)")

agree2_df    = cmp2[cmp2["agree"]]
disagree2_df = cmp2[~cmp2["agree"]]

fig = plt.figure(figsize=(16, 9))
ax  = make_conus_ax(fig, title=(
    f"AEP: K-Means vs SKATER agreement (optimal label mapping)\n"
    f"ARI={ari2:.3f}  |  agree={n_agree2:,}/{len(cmp2):,} ({100*n_agree2/len(cmp2):.0f}%)  "
    f"|  sil cost={sil_km2 - sil_skater2:+.4f}"
))
ax.scatter(
    agree2_df.longitude, agree2_df.latitude,
    s=15, color=rplot.OKABE_ITO[1], alpha=0.6,
    transform=ccrs.PlateCarree(), zorder=4,
    label=f"Agree ({len(agree2_df):,})"
)
ax.scatter(
    disagree2_df.longitude, disagree2_df.latitude,
    s=10, color=rplot.OKABE_ITO[5], alpha=0.9,
    transform=ccrs.PlateCarree(), zorder=5,
    label=f"Disagree ({len(disagree2_df):,})"
)
ax.legend(markerscale=2, fontsize=9, loc="lower left", framealpha=0.8)
plt.tight_layout()
plt.show()

---
## Summary table

Consolidated comparison of both methods across both feature sets.

In [ ]:
summary = pd.DataFrame([
    {
        "Analysis"  : "LP3",
        "Method"    : "K-Means",
        "k"         : N_CLUSTERS_LP3,
        "Silhouette": round(sil_km, 4),
        "ARI vs SKATER": round(ari1, 3),
        "NMI vs SKATER": round(nmi1, 3),
        "Agreement %": round(100 * n_agree1 / len(cmp1), 1),
    },
    {
        "Analysis"  : "LP3",
        "Method"    : "SKATER",
        "k"         : N_CLUSTERS_LP3,
        "Silhouette": round(sil_skater, 4),
        "ARI vs SKATER": 1.0,
        "NMI vs SKATER": 1.0,
        "Agreement %": 100.0,
    },
    {
        "Analysis"  : "AEP",
        "Method"    : "K-Means",
        "k"         : N_CLUSTERS_AEP,
        "Silhouette": round(sil_km2, 4),
        "ARI vs SKATER": round(ari2, 3),
        "NMI vs SKATER": round(nmi2, 3),
        "Agreement %": round(100 * n_agree2 / len(cmp2), 1),
    },
    {
        "Analysis"  : "AEP",
        "Method"    : "SKATER",
        "k"         : N_CLUSTERS_AEP,
        "Silhouette": round(sil_skater2, 4),
        "ARI vs SKATER": 1.0,
        "NMI vs SKATER": 1.0,
        "Agreement %": 100.0,
    },
])

print(summary.to_string(index=False))